<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week03_transformers/week03_fomc_surprise.ipynb)

# How Surprising Is the Fed? Per-Document Bits per Character from a From-Scratch Tiny Transformer on FOMC Text

**J. Francisco Salazar (FranQuant)** · 2026-09-25 · The AI Engineer, Week 3 capstone · Pre-registration: `DESIGN.md`

**Abstract.** How surprising is each FOMC document to a language model that has read only earlier FOMC text? A tiny decoder-only transformer, built from scratch, is trained on statements and minutes public by the end of 2019 and scores each later document in bits per character (BPC), split by meeting into near (2020–2021) and far (2022 onward). Pre-registered hypotheses: far documents are more surprising than near ones (H1); minutes are more surprising than their meeting's statement (H2); the ranking survives a BPE tokenizer (H3). In the confirmatory run H1 is contradicted; H2 and H3 are supported.

**Setup.** Clones the repository at `REF` if needed, requires a CUDA GPU (smoke mode excepted), refuses a real run on code not freshly cloned at `REF`, prints library versions. Only `MODE` differs between modes.

In [ ]:
import time

T_START = time.perf_counter()
TIMES = {}  # stage -> seconds, reported in section 9

import dataclasses  # noqa: E402
import json  # noqa: E402
import math  # noqa: E402
import platform  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
from datetime import date  # noqa: E402
from pathlib import Path  # noqa: E402

import torch  # noqa: E402

MODE = "real"
REF = "week03-v2"
DESIGN_VERSION = "v0.8"
# MODE: "real" is the only mode that scores real N/F. "rehearsal" (CUDA,
# full frozen settings) and "smoke" (CPU only, tiny settings) replace N and
# F by FAKE splits cut from T meetings; their numbers are not results.
MODES = ("real", "rehearsal", "smoke")
if MODE not in MODES:
    raise ValueError(f"MODE must be one of {MODES}")
FAKE_SPLIT = MODE != "real"
SMOKE = MODE == "smoke"
if torch.cuda.is_available():
    if SMOKE:
        raise RuntimeError("smoke mode is CPU-only; use MODE = 'rehearsal' "
                           "on a GPU")
    DEVICE = torch.device("cuda")
elif SMOKE:
    DEVICE = torch.device("cpu")
else:
    raise RuntimeError("CUDA GPU required: Runtime -> Change runtime type "
                       "-> T4 GPU, then Run all.")

REPO_URL = "https://github.com/FranQuant/the-ai-engineer.git"
SUBDIR = Path("capstones/week03_transformers")
MODULES = ("data.py", "bpe.py", "model.py", "evaluate.py", "ngram.py",
           "train.py", "analysis.py")


def has_modules(p: Path) -> bool:
    return all((p / m).is_file() for m in MODULES)


cwd = Path.cwd()
W3 = next((p.resolve() for p in (cwd, cwd.parent, cwd / SUBDIR,
                                 cwd / "the-ai-engineer" / SUBDIR)
           if has_modules(p)), None)
CODE_SOURCE = "local"  # "clone" when the modules come from the clone below
if W3 is None:
    dest = cwd / "the-ai-engineer"
    if dest.exists():
        raise RuntimeError(f"{dest} exists but lacks {MODULES}; remove it")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    REPO_URL, str(dest)], check=True)
    W3 = (dest / SUBDIR).resolve()
    if not has_modules(W3):
        raise RuntimeError(f"clone lacks {MODULES} under {SUBDIR}")
    CODE_SOURCE = "clone"


def uncommitted_changes(path: Path):
    """True if `git status --porcelain` lists changes under `path`, False if
    it lists none, None if git or the repository is unavailable."""
    try:
        out = subprocess.run(["git", "-C", str(path), "status",
                              "--porcelain", "--", "."],
                             capture_output=True, text=True)
    except FileNotFoundError:
        return None
    if out.returncode != 0:
        return None
    return bool(out.stdout.strip())


def real_mode_refusal(path: Path, code_source: str, ref: str):
    """Why a real run must not use the code in `path`, or None if it may:
    this session cloned it, and its HEAD is the commit `ref` names."""
    fix = ("Use Runtime → Disconnect and delete runtime, then Run all "
           "again.")
    if code_source != "clone":
        return (f"real mode needs code this session cloned at {ref}, but "
                f"found existing modules in {path}. {fix}")
    head, target = (subprocess.run(["git", "-C", str(path), "rev-parse", r],
                                   capture_output=True, text=True)
                    for r in ("HEAD", f"{ref}^{{commit}}"))
    if head.returncode or target.returncode or head.stdout != target.stdout:
        head, target = (r.stdout.strip() if r.returncode == 0
                        else "unresolved" for r in (head, target))
        return (f"real mode needs HEAD in {path} at {ref}, but HEAD is "
                f"{head} and {ref} is {target}. {fix}")
    return None


UNCOMMITTED_CHANGES = uncommitted_changes(W3)
if MODE == "real":
    refusal = real_mode_refusal(W3, CODE_SOURCE, REF)
    if refusal:
        raise RuntimeError(refusal)
sys.path.insert(0, str(W3))

import analysis  # noqa: E402
import bpe  # noqa: E402
import data  # noqa: E402
import evaluate  # noqa: E402
import model  # noqa: E402
import ngram  # noqa: E402
import train  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

GIT_COMMIT = subprocess.run(["git", "-C", str(W3), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda,
       "gpu": (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else f"none ({DEVICE.type})"),
       "ref": REF, "git_commit": GIT_COMMIT, "code_source": CODE_SOURCE,
       "uncommitted_changes": UNCOMMITTED_CHANGES, "mode": MODE,
       "design_version": DESIGN_VERSION}
for k, v in ENV.items():
    print(f"{k:>14}: {v}")
RUN_DIR = W3 / "runs" / MODE  # each mode has its own directory
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BANNER = ("#" * 72 + f"\n#  {MODE.upper()} RUN: fake N and F cut from T "
          "meetings. NOT A RESULT.\n" + "#" * 72)
if FAKE_SPLIT:
    print("\n" + BANNER)
TIMES["setup"] = time.perf_counter() - T_START

**Numbers behind the abstract**, from `runs/confirmatory_results.json` (the first real run, tag `week03-v2-run1`), the source of every confirmatory number below.

In [ ]:
# The confirmatory run (DESIGN §9): the first real run, archived after it.
CONF_PATH = W3 / "runs" / "confirmatory_results.json"
if not CONF_PATH.is_file():
    raise FileNotFoundError(
        f"{CONF_PATH} is missing: REF must name a commit that includes the "
        "archived confirmatory run")
CONF = json.loads(CONF_PATH.read_text())
assert CONF["mode"] == "real"
assert CONF["environment"]["ref"] == "week03-v2-run1"
CR = CONF["results"]
CONF_SCORES = {k: [evaluate.DocumentScore(**d) for d in v]
               for k, v in CONF["scores"].items()}


def fmt_estimate(e):
    """Point, 95% CI and label of an Estimate.as_dict() record."""
    label = f" -> {e['label'].upper()}" if e["label"] else ""
    return (f"{e['point']:+.4f}, 95% CI [{e['ci_low']:+.4f}, "
            f"{e['ci_high']:+.4f}]{label}")


env = CONF["environment"]
print(f"Confirmatory run: tag {env['ref']}, commit {env['git_commit'][:7]}, "
      f"{env['gpu']}, torch {env['torch']}")
for key, what in (("H1", "Δ₁ = mean BPC(F) − mean BPC(N)"),
                  ("H2", "Δ₂ = mean [BPC(minutes) − BPC(statement)], "
                         "matched F meetings"),
                  ("H3", "Spearman ρ(char, BPE) on F")):
    print(f"{key}: {what} = {fmt_estimate(CR[key])}")
means = CONF["mean_bpc"]["char"]
print(f"mean char BPC: N {means['N']:.4f} "
      f"({CONF['counts']['N']['documents']} documents), "
      f"F {means['F']:.4f} ({CONF['counts']['F']['documents']} documents)")
assert [CR[k]["label"] for k in ("H1", "H2", "H3")] == [
    "contradicted", "supported", "supported"]  # as stated in the abstract

## 1. Introduction

The FOMC publishes post-meeting statements and, for scheduled meetings, minutes. A language model trained only on earlier FOMC text measures how new each document's wording is, in bits per character (BPC): a textual, not a market, measure.

**Pre-registration.** `DESIGN.md` fixed split, scoring, statistics, labels and compute before training on the real split; the first real run (tag `week03-v2-run1`) is confirmatory.

**Split** by meeting: T (train) available by 2019-12-31; N (near) 2020–2021; F (far) from 2022.

- **H1 (drift):** Δ₁ = mean char BPC(F) − mean char BPC(N) > 0.
- **H2 (genre):** Δ₂ = mean over matched F meetings of char BPC(minutes) − BPC(statement) > 0; statements are expected to be more formulaic.
- **H3 (instrument robustness):** Spearman ρ(char, BPE transformer) on F ≥ 0.7.

**Labels** (meeting bootstrap, 2,000 resamples): *supported* if the 95% CI excludes 0 in the predicted direction; *contradicted* if the point estimate is ≤ 0; otherwise *inconclusive*. H3 uses the threshold alone.

**Exploratory:** E1, the five highest-BPC F documents; E2, ρ(char transformer, 5-gram) on F.

## 2. Data, normalization, split

The corpus, its manifest and the split manifest must match hard-coded SHA-256 values. Normalization follows DESIGN §3; the split is the committed meeting-level manifest. Every check fails closed.

In [ ]:
t0 = time.perf_counter()
_docs = data.load_documents()  # SHA-256 + length checks; fails closed
split_manifest = data.load_split_manifest()
rules = split_manifest["rules"]
# Document IDs and real body sizes per split, from the manifest (no text).
MANIFEST_IDS = {sp: {e["document_id"] for m in split_manifest["meetings"]
                     if m["split"] == sp for e in m["documents"]}
                for sp in data.SPLITS}
REAL_CHARS = {k: sum(e["body_code_points"] for m in split_manifest["meetings"]
                     if m["split"] in sps for e in m["documents"])
              for k, sps in (("N+F", ("N", "F")), ("F", ("F",)))}

if FAKE_SPLIT:
    # N and F bodies are dropped right after the load and hash checks.
    t_only = [d for d in _docs if d.split == "T"]
    del _docs
    # FAKE split from T meetings only: the last 20% of T meetings as fake
    # F, the 10% before them as fake N, the rest as training T.
    t_meetings = sorted({d.meeting for d in t_only})
    n_fake_f = round(0.2 * len(t_meetings))
    n_fake_n = round(0.1 * len(t_meetings))
    fake_f = set(t_meetings[len(t_meetings) - n_fake_f:])
    fake_n = set(t_meetings[len(t_meetings) - n_fake_f - n_fake_n:
                            len(t_meetings) - n_fake_f])
    docs = [dataclasses.replace(
        d, split="F" if d.meeting in fake_f else
        "N" if d.meeting in fake_n else "T") for d in t_only]
    del t_only
    last_n = date.fromisoformat(max(fake_n))
    first_f = date.fromisoformat(min(fake_f))
    NF_BOUNDARY = last_n + (first_f - last_n) / 2
else:
    docs = [d for d in _docs if d.split in ("T", "N", "F")]
    del _docs
    NF_BOUNDARY = date.fromisoformat(rules["near_end"])
if data.check_characters(docs):  # §3: counts only, no text
    raise RuntimeError("INFEASIBLE (§10): N/F characters absent from T")

train_docs = [d for d in docs if d.split == "T"]
n_docs = [d for d in docs if d.split == "N"]
f_docs = [d for d in docs if d.split == "F"]


def split_counts(ds):
    by_meeting = {}
    for d in ds:
        by_meeting.setdefault(d.meeting, set()).add(d.genre)
    return {"documents": len(ds), "meetings": len(by_meeting),
            "statements": sum(d.genre == "statement" for d in ds),
            "minutes": sum(d.genre == "minutes" for d in ds),
            "matched_meetings": sum(g == {"statement", "minutes"}
                                    for g in by_meeting.values()),
            "body_chars": sum(len(d.body) for d in ds)}


COUNTS = {s: split_counts(ds) for s, ds in
          (("T", train_docs), ("N", n_docs), ("F", f_docs))}
print(f"{'split':>5} " + " ".join(f"{k:>16}" for k in COUNTS["T"]))
for s, c in COUNTS.items():
    print(f"{s:>5} " + " ".join(f"{v:>16,}" for v in c.values()))
print(f"N/F boundary for plots: {NF_BOUNDARY}")

# §2 feasibility thresholds (the real split only; the fake one is smaller)
FEASIBILITY = {"F documents >= 40": COUNTS["F"]["documents"] >= 40,
               "F matched pairs >= 15": COUNTS["F"]["matched_meetings"] >= 15,
               "N documents >= 16": COUNTS["N"]["documents"] >= 16}
if FAKE_SPLIT:
    print("§2 thresholds: not applied to the fake split")
else:
    for k, ok in FEASIBILITY.items():
        print(f"§2 {k}: {'pass' if ok else 'FAIL'}")
    if not all(FEASIBILITY.values()):
        raise RuntimeError("INFEASIBLE (§10): §2 thresholds fail")
print("§3 character check: pass")


def assert_scorable(ds, splits):
    """Documents (or their scores) are in `splits`. On a fake split every
    one is a real T document by manifest ID, never a real N or F one."""
    ds = list(ds)
    assert {d.split for d in ds} <= set(splits), "unexpected split"
    if FAKE_SPLIT:
        ids = {d.id for d in ds}
        assert not ids & (MANIFEST_IDS["N"] | MANIFEST_IDS["F"]), (
            "real N/F document on a fake split")
        assert ids <= MANIFEST_IDS["T"], "fake split holds a non-T document"


TIMES["data"] = time.perf_counter() - t0

## 3. Model and checks

`model.py` implements the transformer from scratch: masked attention, multi-head attention, FFN, Pre-LN blocks, sinusoidal positions. Attention and MHA are hand-written and tested; training uses PyTorch's fused SDPA after the numeric parity checks below. The verification checks (DESIGN §11, handout Appendix B) run here on CPU.

In [ ]:
t0 = time.perf_counter()
torch.set_printoptions(precision=4, sci_mode=False)
sdpa = model.scaled_dot_product_attention
CHECKS = {}  # every value printed below, saved in results.json

# §4.3 worked example (Appendix B): S = Q K^T / sqrt(d_k)
Q = torch.tensor([[1., 0], [0, 1], [1, 1]])
K = torch.tensor([[1., 0], [1, 1], [0, 1]])
V = torch.tensor([[1., 0], [0, 2], [3, 1]])
QKT = Q @ K.T
S_raw = QKT / math.sqrt(2)
Y, S, A = sdpa(Q, K, V)
print("Q K^T =", QKT, "S = Q K^T / sqrt(2) =", S_raw, "A = softmax(S) =", A,
      "Y = A V =", Y, sep="\n")
r, e = 1 / math.sqrt(2), math.exp(-1 / math.sqrt(2))
assert torch.equal(QKT, torch.tensor([[1., 1, 0], [0, 1, 1], [1, 2, 1]]))
assert torch.allclose(S_raw, r * QKT, atol=1e-6)
# sdpa returns S shifted by its row max (softmax-invariant).
assert torch.allclose(S, S_raw - S_raw.max(dim=-1, keepdim=True).values,
                      atol=1e-6)
A_ref = torch.tensor([[1, 1, e], [e, 1, 1], [e, 1, e]])
assert torch.allclose(A, A_ref / A_ref.sum(-1, keepdim=True), atol=1e-6)
assert torch.allclose(Y, torch.tensor([[0.994440, 1.0], [1.401112, 1.203336],
                                       [0.993020, 1.255235]]), atol=1e-5)
CHECKS["worked_example"] = {"QKT": QKT.tolist(), "S": S_raw.tolist(),
                            "S_shifted": S.tolist(), "A": A.tolist(),
                            "Y": Y.tolist()}

# Causal rerun: token i attends to tokens <= i only.
Yc, _, Ac = sdpa(Q, K, V, mask=model.make_causal_mask(3))
print("causal A =", Ac, "causal Y =", Yc, sep="\n")
assert torch.allclose(Ac, torch.tensor([[1., 0, 0], [0.330238, 0.669762, 0],
                                        [0.248255, 0.503490, 0.248255]]),
                      atol=1e-5)
assert torch.allclose(Yc, torch.tensor([[1., 0], [0.330238, 1.339523],
                                        [0.993020, 1.255235]]), atol=1e-5)
Ya, _, _ = sdpa(Q, K, V, mask=model.make_causal_mask(3, additive=True))
assert torch.equal(Ya, Yc)  # boolean and additive masks agree
CHECKS["causal"] = {"A": Ac.tolist(), "Y": Yc.tolist()}

# Fused (PyTorch) vs manual attention
g = torch.Generator().manual_seed(0)
q, k, v = (torch.randn(2, 4, 16, 8, generator=g) for _ in range(3))
for causal in (False, True):
    manual, _, _ = sdpa(q, k, v, mask=model.make_causal_mask(16)
                        if causal else None)
    fused = torch.nn.functional.scaled_dot_product_attention(
        q, k, v, is_causal=causal)
    diff = (manual - fused).abs().max().item()
    print(f"fused vs manual (causal={causal}): max |diff| = {diff:.2e}")
    assert diff < 1e-5
    CHECKS[f"fused_vs_manual_max_diff_causal_{causal}"] = diff

# MHA with H = 1 equals single-head self-attention
torch.manual_seed(3)
sa = model.SelfAttention(d_model=4, causal=True)
mha = model.MultiHeadAttention(d_model=4, num_heads=1, causal=True)
with torch.no_grad():
    mha.proj_qkv.weight.copy_(torch.cat(
        [sa.W_Q.weight, sa.W_K.weight, sa.W_V.weight]))
    mha.proj_out.weight.copy_(torch.eye(4))
x = torch.randn(1, 5, 4)
diff = (mha(x) - sa(x)).abs().max().item()
print(f"MHA(H=1) vs self-attention: max |diff| = {diff:.2e}")
assert diff < 1e-6
CHECKS["mha_h1_vs_self_attention_max_diff"] = diff

# Appendix B tiny block: B = 1, T = 3, d_model = 4, H = 2, d_ff = 8, with
# simple weights; checked against a manual pass that slices the heads.
D, H, T3, DH = 4, 2, 3, 2
blk = model.TransformerBlock(D, H, d_ff=8, causal=True)
with torch.no_grad():
    for name, p in blk.named_parameters():
        if p.dim() == 2:  # linear weights: values in {-0.2, ..., 0.2}
            p.copy_(((torch.arange(p.numel()) * 3) % 5 - 2)
                    .reshape(p.shape) * 0.1)
        else:  # layer norm weight 1, every bias 0
            p.fill_(1.0 if name.startswith("ln") and name.endswith("weight")
                    else 0.0)
x0 = torch.eye(T3, D).unsqueeze(0)  # three one-hot "tokens"
xb = model.PositionalEncoding(D, max_len=T3)(x0)
captured = {}
hook = blk.attn.proj_qkv.register_forward_hook(
    lambda mod, inp, out: captured.update(qkv=out))
with torch.no_grad():
    out = blk(xb)
hook.remove()
Qh, Kh, Vh = (t.transpose(1, 2) for t in captured["qkv"].view(
    1, T3, 3, H, DH).unbind(dim=2))
_, _, A_blk = sdpa(Qh, Kh, Vh, mask=model.make_causal_mask(T3))


def layer_norm(z, ln):
    mu = z.mean(-1, keepdim=True)
    var = z.var(-1, unbiased=False, keepdim=True)
    return (z - mu) / torch.sqrt(var + ln.eps) * ln.weight + ln.bias


with torch.no_grad():
    pos = torch.arange(T3).float()[:, None]
    freq = 10000.0 ** (-torch.arange(0, D, 2).float() / D)
    pe = torch.stack([torch.sin(pos * freq), torch.cos(pos * freq)],
                     dim=-1).reshape(T3, D)  # sin, cos interleaved
    xm = x0[0] + pe
    qkv = layer_norm(xm, blk.ln1) @ blk.attn.proj_qkv.weight.T
    Qm, Km, Vm = qkv[:, :D], qkv[:, D:2 * D], qkv[:, 2 * D:]
    keep = torch.tril(torch.ones(T3, T3, dtype=torch.bool))
    A_man, heads = [], []
    for h in range(H):
        cols = slice(h * DH, (h + 1) * DH)
        s = Qm[:, cols] @ Km[:, cols].T / math.sqrt(DH)
        A_man.append(torch.softmax(s.masked_fill(~keep, float("-inf")), -1))
        heads.append(A_man[-1] @ Vm[:, cols])
    x1 = xm + torch.cat(heads, dim=-1) @ blk.attn.proj_out.weight.T
    lin1, lin2 = blk.ff.net[0], blk.ff.net[2]
    z = layer_norm(x1, blk.ln2) @ lin1.weight.T + lin1.bias
    y_man = x1 + (0.5 * z * (1 + torch.erf(z / math.sqrt(2)))
                  @ lin2.weight.T + lin2.bias)
print("tiny block Q =", Qm, "K =", Km, "V =", Vm,
      "attention weights (per head) =", A_blk[0], "block output =", out[0],
      sep="\n")
assert xb.shape == out.shape == (1, T3, D)
assert A_blk.shape == (1, H, T3, T3)
assert torch.allclose(A_blk.sum(-1), torch.ones(1, H, T3), atol=1e-6)
assert torch.allclose(xb[0], xm, atol=1e-6)  # positional encoding
assert torch.allclose(A_blk[0], torch.stack(A_man), atol=1e-6)
diff = (out[0] - y_man).abs().max().item()
print(f"tiny block vs manual: max |diff| = {diff:.2e}")
assert diff < 1e-6
CHECKS["tiny_block"] = {"attention": A_blk[0].tolist(),
                        "output": out[0].tolist(), "max_diff_vs_manual": diff}

# Uniform logits give loss log V and near-uniform samples
lm = model.TinyTransformerLM(model.ModelConfig(
    vocab_size=87, d_model=16, num_heads=2, num_layers=1, d_ff=32,
    block_size=8, dropout=0.0))
with torch.no_grad():
    lm.tok_emb.weight.zero_()  # tied head: every logit is 0
_, loss = lm(torch.randint(0, 87, (4, 8)), torch.randint(0, 87, (4, 8)))
print(f"uniform logits: loss {loss.item():.6f}, log V {math.log(87):.6f}")
assert abs(loss.item() - math.log(87)) < 1e-6
# 50 x 500 = 25,000 tokens: about 287 per token, so ±30% is about 5 sd.
sample = lm.generate(torch.zeros(50, 1, dtype=torch.long), 500,
                     generator=torch.Generator().manual_seed(0))[:, 1:]
rel = torch.bincount(sample.flatten(), minlength=87) / sample.numel() * 87
print(f"uniform sample: {sample.numel():,} tokens, frequency x V in "
      f"[{rel.min():.3f}, {rel.max():.3f}]")
assert ((rel - 1).abs() <= 0.3).all()
CHECKS["uniform_logits"] = {"loss": loss.item(), "log_V": math.log(87),
                            "sample_tokens": sample.numel(),
                            "sample_freq_x_V_range": [rel.min().item(),
                                                      rel.max().item()]}

# Trivial-pattern overfit: ABAB... is learned to near-zero loss
torch.manual_seed(1)
ab = torch.tensor([0, 1] * 200)
lm = model.TinyTransformerLM(model.ModelConfig(
    vocab_size=2, d_model=16, num_heads=2, num_layers=2, d_ff=32,
    block_size=8, dropout=0.0))
opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
for _ in range(300):
    ix = torch.randint(0, len(ab) - 9, (16,))
    xb = torch.stack([ab[i:i + 8] for i in ix])
    yb = torch.stack([ab[i + 1:i + 9] for i in ix])
    _, loss = lm(xb, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
ab_ids = lm.generate(torch.tensor([[0]]), 20, greedy=True)[0].tolist()
cont = "".join("AB"[i] for i in ab_ids)
print(f"AB overfit: loss after 300 steps {loss.item():.4f}; greedy from "
      f"'A': {cont}")
assert loss.item() < 0.05
assert cont == "AB" * 10 + "A"
CHECKS["ab_overfit_loss_after_300_steps"] = loss.item()
CHECKS["ab_greedy_from_A"] = cont
del lm, mha, sa, opt, blk
TIMES["checks"] = time.perf_counter() - t0
print(f"all §11 and Appendix B checks passed ({TIMES['checks']:.1f} s)")

## 4. Training

Frozen settings (DESIGN §8), printed by the code. Tokenizers and transformers see T only; N feeds the training curves, which no decision reads.

In [ ]:
if SMOKE:
    ARCH = {"d_model": 64, "num_layers": 2, "num_heads": 4, "d_ff": 128}
    BLOCK_SIZE, BPE_VOCAB = 128, 500
    CFG = train.TrainConfig(steps=60, batch_size=8, warmup_steps=10,
                            monitor_batches=2, monitor_batch_size=8)
else:
    ARCH, BLOCK_SIZE, BPE_VOCAB = train.ARCH, train.BLOCK_SIZE, train.BPE_VOCAB
    CFG = train.TrainConfig()
_probe = train.make_model(10, BLOCK_SIZE, CFG.seed, **ARCH)
SETTINGS = {"arch": ARCH, "block_size": BLOCK_SIZE, "bpe_vocab": BPE_VOCAB,
            "train": dataclasses.asdict(CFG),
            "precision": "fp16" if DEVICE.type == "cuda" else "fp32",
            "dropout": _probe.blocks[0].ff.net[-1].p,
            "token_embedding_init_std": ARCH["d_model"] ** -0.5,
            "ngram_order": ngram.ORDER}
del _probe
print(json.dumps(SETTINGS, indent=1))

t0 = time.perf_counter()
char_vocab = data.CharVocab.from_documents(train_docs)
TIMES["char_vocab"] = time.perf_counter() - t0
t0 = time.perf_counter()
bpe_tok = bpe.SimpleBPE.from_documents(train_docs, BPE_VOCAB)
TIMES["bpe_fit"] = time.perf_counter() - t0
assert bpe_tok.vocab_size == BPE_VOCAB
TOKENIZERS = {"char": char_vocab, "bpe": bpe_tok}

t0 = time.perf_counter()
assert_scorable(n_docs, ("N",))
samplers, monitors, TOKEN_STATS = {}, {}, {}
for name, tok in TOKENIZERS.items():
    samplers[name] = data.WindowSampler.from_documents(train_docs, tok,
                                                       BLOCK_SIZE)
    monitors[name] = data.NMonitorSampler.from_documents(n_docs, tok,
                                                         BLOCK_SIZE)
    n_tokens = int(samplers[name].data.numel())
    TOKEN_STATS[name] = {
        "vocab_size": tok.vocab_size, "t_tokens_serialized": n_tokens,
        "tokens_per_char": (n_tokens - data.PREFIX_LEN * len(train_docs))
        / COUNTS["T"]["body_chars"],
        "passes_over_t": CFG.steps * CFG.batch_size * BLOCK_SIZE / n_tokens}
TIMES["tokenize"] = time.perf_counter() - t0
for name, s in TOKEN_STATS.items():
    print(f"{name:>4}: vocab {s['vocab_size']:,}, {s['t_tokens_serialized']:,}"
          f" T tokens ({s['tokens_per_char']:.4f}/char), "
          f"{s['passes_over_t']:.2f} passes over T")
print(f"BPE fit: {TIMES['bpe_fit']:.1f} s")

Char transformer: fixed step count, final weights kept, no model selection.

In [ ]:
MODELS, TRAINING = {}, {}


def run(name):
    tok = TOKENIZERS[name]
    net = train.make_model(tok.vocab_size, BLOCK_SIZE, CFG.seed, **ARCH)
    n_params = sum(p.numel() for p in net.parameters())
    init_std = net.tok_emb.weight.std().item()
    print(f"--- {name} transformer: {n_params:,} parameters, token "
          f"embedding init std {init_std:.4f}")
    res = train.train(net, samplers[name], monitors[name], CFG, DEVICE,
                      log_every=max(1, CFG.steps // 10))
    MODELS[name] = net.eval()
    TRAINING[name] = {"n_params": n_params,
                      "token_embedding_init_std_measured": init_std,
                      **dataclasses.asdict(res)}
    TIMES[f"train_{name}"] = res.seconds
    tail = res.train_loss[-50:]
    print(f"{name}: {res.steps:,} steps in {res.seconds:.1f} s "
          f"({res.precision}); mean train loss over the last {len(tail)} "
          f"steps {sum(tail) / len(tail):.4f} nats/token; "
          f"{res.skipped_updates} GradScaler-skipped updates (§8: counted, "
          f"not replaced)")


run("char")

BPE transformer: same recipe; only the tokenizer and the vocabulary-dependent parameters differ.

In [ ]:
run("bpe")

Sampling demo, greedy and at temperature 0.8. Qualitative; nothing downstream reads it.

In [ ]:
# One short sampling demo from the char model (§11): greedy, then T = 0.8.
t0 = time.perf_counter()
PROMPT = "The Committee"
prompt_ids = torch.tensor([data.serialize(char_vocab, "statement", PROMPT)],
                          device=DEVICE)
gen = torch.Generator(device=DEVICE).manual_seed(CFG.seed)
SAMPLES = {}
for label, kwargs in (("greedy", {"greedy": True}),
                      ("temperature 0.8", {"temperature": 0.8,
                                           "generator": gen})):
    out = MODELS["char"].generate(prompt_ids, 200, **kwargs)
    SAMPLES[label] = char_vocab.decode(out[0, data.PREFIX_LEN:].tolist())
    print(f"[{label}]\n{SAMPLES[label]}\n")
TIMES["sampling"] = time.perf_counter() - t0

## 5. Scoring

Each document is scored alone from `<BOS>` + genre, half-overlapping windows, every body target once, fp32 (DESIGN §4). BPC = NLL / (characters × ln 2). Char scores N and F; BPE and 5-gram F.

In [ ]:
assert_scorable(n_docs + f_docs, ("N", "F"))
t0 = time.perf_counter()
SCORES = {"char": evaluate.score_documents(MODELS["char"], char_vocab,
                                           n_docs + f_docs, BLOCK_SIZE)}
TIMES["score_char"] = time.perf_counter() - t0
t0 = time.perf_counter()
SCORES["bpe"] = evaluate.score_documents(MODELS["bpe"], bpe_tok, f_docs,
                                         BLOCK_SIZE)
TIMES["score_bpe"] = time.perf_counter() - t0
t0 = time.perf_counter()
ngram_lm = ngram.WittenBellNgram.from_documents(train_docs)
TIMES["ngram_fit"] = time.perf_counter() - t0
t0 = time.perf_counter()
SCORES["ngram"] = ngram_lm.score_documents(f_docs)
TIMES["score_ngram"] = time.perf_counter() - t0

for name, scores in SCORES.items():
    assert_scorable(scores, ("N", "F"))  # DocumentScore has id and split
    assert len({s.id for s in scores}) == len(scores)
if FAKE_SPLIT:
    print(f"{MODE} guard: every scored document is a real T document "
          "(manifest IDs); no real N/F document was scored")

MEAN_BPC = {name: {sp: float(np.mean([s.bpc for s in scores
                                      if s.split == sp]))
                   for sp in ("N", "F") if any(s.split == sp
                                               for s in scores)}
            for name, scores in SCORES.items()}
for name, means in MEAN_BPC.items():
    print(f"{name:>5}: " + ", ".join(
        f"{sp} {len([s for s in SCORES[name] if s.split == sp])} docs, "
        f"mean BPC {m:.4f}" for sp, m in means.items())
        + f"  ({TIMES['score_' + name]:.1f} s)")

# Rehearsal: scoring time projected to the real N+F (char) and F (BPE,
# n-gram) body sizes from the manifest, at the rate measured here.
SCORING_PROJECTION = None
if FAKE_SPLIT:
    fake_chars = {"N+F": COUNTS["N"]["body_chars"] + COUNTS["F"]["body_chars"],
                  "F": COUNTS["F"]["body_chars"]}
    SCORING_PROJECTION = {}
    for name, target in (("char", "N+F"), ("bpe", "F"), ("ngram", "F")):
        rate = TIMES[f"score_{name}"] / fake_chars[target]
        SCORING_PROJECTION[name] = {
            "target": target, "fake_chars": fake_chars[target],
            "real_chars": REAL_CHARS[target], "measured_s":
            TIMES[f"score_{name}"], "projected_s": rate * REAL_CHARS[target]}
        print(f"projected {name} scoring on real {target} "
              f"({REAL_CHARS[target]:,} chars): "
              f"{SCORING_PROJECTION[name]['projected_s']:.1f} s")

## 6. Results

The frozen `analysis.py` computes this run's statistics. Subsections print the confirmatory value beside this run's; prose and `assert` lines refer to the confirmatory value.

In [ ]:
t0 = time.perf_counter()
H1 = analysis.h1_drift(SCORES["char"])
H2 = analysis.h2_genre(SCORES["char"])
H3 = analysis.h3_instruments(SCORES["char"], SCORES["bpe"])
E1 = analysis.e1_top(SCORES["char"])
E2 = analysis.e2_ngram(SCORES["char"], SCORES["ngram"])
TIMES["statistics"] = time.perf_counter() - t0
THIS_RUN = {"H1": H1.as_dict(), "H2": H2.as_dict(), "H3": H3.as_dict(),
            "E2": E2.as_dict()}
CONF_LABEL = f"confirmatory ({CONF['environment']['ref']})"
RUN_LABEL = (f"this run ({MODE}, FAKE split)" if FAKE_SPLIT
             else f"this run ({MODE}, {GIT_COMMIT[:7]})")
NOTE = ("Differences between runs are expected (fp16 GPU training is not "
        "bit-reproducible); the confirmatory run is the result (DESIGN §9).")


def compare(key, what):
    """Print one statistic for the confirmatory run and for this run."""
    print(f"{key}: {what}")
    for label, res in ((CONF_LABEL, CR), (RUN_LABEL, THIS_RUN)):
        e = res[key]
        print(f"  {label:<34} {fmt_estimate(e)}  "
              f"({e['n_documents']} documents, {e['n_meetings']} meetings)")
    print(NOTE)
    if FAKE_SPLIT:
        print(f"This run is {MODE.upper()} on a fake split: NOT A RESULT.")


print(f"statistics computed in {TIMES['statistics']:.1f} s")

Figures of this run, saved under `runs/<MODE>/figures/`: training curves, BPC over time, statement vs minutes, char vs BPE.

In [ ]:
t0 = time.perf_counter()
# Reference palette, slots 1-2 (validated pair); text in ink tokens.
C1, C2 = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "axes.titlecolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 10})
FIGURES = {}


def save(fig, name):
    if FAKE_SPLIT:
        fig.text(0.5, 0.5, f"{MODE.upper()}: NOT A RESULT", ha="center",
                 va="center", fontsize=28, color=INK2, alpha=0.25,
                 rotation=20)
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    FIGURES[name] = str(path)
    plt.show()


# Figure 1: training curves (T training loss, N monitoring) per instrument
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, name in zip(axes, ("char", "bpe")):
    tr = TRAINING[name]
    loss = np.array(tr["train_loss"])
    w = max(1, len(loss) // 50)
    smooth = np.convolve(loss, np.ones(w) / w, mode="valid")
    ax.plot(np.arange(w, len(loss) + 1), smooth, color=C1,
            label=f"T training loss ({w}-step mean)")
    ax.plot([m["step"] for m in tr["monitor"]],
            [m["loss"] for m in tr["monitor"]], color=C2, marker="o",
            markersize=6, label="N monitoring (plot only)")
    ax.set(title=f"{name} transformer", xlabel="step",
           ylabel=f"loss (nats per {name} token)")
axes[0].legend()
save(fig, "fig1_training_curves")

# Figure 2: per-document char BPC over time, N and F
fig, ax = plt.subplots(figsize=(10, 3.8))
for sp, color in (("N", C1), ("F", C2)):
    for genre, marker in (("statement", "o"), ("minutes", "s")):
        pts = [(date.fromisoformat(s.meeting), s.bpc) for s in SCORES["char"]
               if s.split == sp and s.genre == genre]
        if pts:
            ax.scatter(*zip(*pts), color=color, marker=marker, s=36,
                       edgecolors="white", linewidths=0.8,
                       label=f"{sp} {genre}")
ax.axvline(NF_BOUNDARY, color=INK2, linestyle="--", linewidth=1)
ax.annotate("N | F boundary", (NF_BOUNDARY, 1), xycoords=("data",
            "axes fraction"), xytext=(4, -12), textcoords="offset points",
            color=INK2, fontsize=9)
ax.set(title="Char transformer BPC per document", xlabel="meeting date",
       ylabel="bits per character")
ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, -0.18))
save(fig, "fig2_bpc_over_time")

# Figure 3: paired genre plot over matched F meetings (H2)
pairs, _ = analysis.genre_pairs(SCORES["char"])
fig, ax = plt.subplots(figsize=(4.5, 4))
for _, m_bpc, s_bpc in pairs:
    ax.plot([0, 1], [s_bpc, m_bpc], color=GRID, linewidth=1, zorder=1)
ax.scatter([0] * len(pairs), [s for _, _, s in pairs], color=C1, s=36,
           zorder=2, label="statement")
ax.scatter([1] * len(pairs), [m for _, m, _ in pairs], color=C2, s=36,
           zorder=2, label="minutes")
ax.set(xticks=[0, 1], xticklabels=["statement", "minutes"], xlim=(-0.4, 1.4),
       title=f"Matched F meetings (n = {len(pairs)})",
       ylabel="char BPC")
ax.legend(loc="upper center")
save(fig, "fig3_genre_pairs")

# Figure 4: char vs BPE BPC on F (H3)
bpe_by_id = {s.id: s.bpc for s in SCORES["bpe"]}
fig, ax = plt.subplots(figsize=(4.8, 4.2))
for genre, color in (("statement", C1), ("minutes", C2)):
    pts = [(s.bpc, bpe_by_id[s.id]) for s in SCORES["char"]
           if s.split == "F" and s.genre == genre]
    ax.scatter(*zip(*pts), color=color, s=36, edgecolors="white",
               linewidths=0.8, label=genre)
ax.set(title=f"F documents: Spearman rho = {H3.point:.3f}",
       xlabel="char transformer BPC", ylabel="BPE transformer BPC")
ax.legend()
save(fig, "fig4_char_vs_bpe")
TIMES["figures"] = time.perf_counter() - t0

### 6.1 H1: drift

Δ₁ with CI and label, then mean char BPC on N and F.

In [ ]:
compare("H1", "Δ₁ = mean BPC(F) − mean BPC(N), char transformer")
for label, means in ((CONF_LABEL, CONF["mean_bpc"]["char"]),
                     (RUN_LABEL, MEAN_BPC["char"])):
    print(f"  {label:<34} mean BPC N {means['N']:.4f}, F {means['F']:.4f}")
h1 = CR["H1"]
assert h1["point"] <= 0 and h1["label"] == "contradicted"
assert h1["ci_low"] < 0 < h1["ci_high"]  # the CI includes 0

**Interpretation.** The confirmatory Δ₁ is ≤ 0, so H1 is *contradicted*: far documents are no more surprising than near ones. The CI includes 0: a small drift of either sign remains compatible. DESIGN §14 anticipated pandemic-era language raising N; section 7 agrees, without changing the label.

### 6.2 H2: genre

Δ₂ with CI and label, its share of statement BPC, and the meetings where minutes score higher.

In [ ]:
compare("H2", "Δ₂ = mean over matched F meetings of "
              "[BPC(minutes) − BPC(statement)]")
for label, scores in ((CONF_LABEL, CONF_SCORES["char"]),
                      (RUN_LABEL, SCORES["char"])):
    pairs, excluded = analysis.genre_pairs(scores)
    m_mean = float(np.mean([m for _, m, _ in pairs]))
    s_mean = float(np.mean([s for _, _, s in pairs]))
    higher = sum(m > s for _, m, s in pairs)
    print(f"  {label:<34} minutes {m_mean:.4f}, statements {s_mean:.4f} "
          f"(Δ₂ = {(m_mean - s_mean) / s_mean:+.1%} of statement BPC); "
          f"minutes higher in {higher} of {len(pairs)} meetings; "
          f"{excluded} unmatched excluded")
h2 = CR["H2"]
assert h2["label"] == "supported" and h2["ci_low"] > 0

**Interpretation.** The confirmatory CI lies above 0, so H2 is *supported*: minutes are more surprising than their meeting's statement, in the printed number of matched meetings. Statements are typically shorter and more formulaic and carry over earlier wording; this design neither measures nor separates these possible explanations.

### 6.3 H3: instrument robustness

ρ(char, BPE) on F with CI, then both mean BPCs on F.

In [ ]:
compare("H3", f"Spearman ρ(char, BPE) on F, threshold "
              f"{analysis.H3_THRESHOLD}")
for label, means in ((CONF_LABEL, CONF["mean_bpc"]),
                     (RUN_LABEL, MEAN_BPC)):
    print(f"  {label:<34} mean BPC on F: char {means['char']['F']:.4f}, "
          f"BPE {means['bpe']['F']:.4f}")
h3 = CR["H3"]
assert h3["label"] == "supported"
assert h3["ci_low"] >= analysis.H3_THRESHOLD  # CI also above threshold
assert CONF["mean_bpc"]["bpe"]["F"] < CONF["mean_bpc"]["char"]["F"]

**Interpretation.** ρ exceeds 0.7, so H3 is *supported*; the lower CI end is also above the threshold. The instruments rank F documents alike despite differing in tokenizer, context and parameters (DESIGN §5). The BPE transformer has the lower mean BPC; H3 compares rankings, and the confounds prevent attributing that gap to tokenization.

### 6.4 Exploratory: E1 and E2

E1: the five highest-BPC F documents. E2: ρ(char transformer, 5-gram) on F. No labels.

In [ ]:
print("E1: top five F documents by char BPC")
for label, top in ((CONF_LABEL, CR["E1"]), (RUN_LABEL, E1)):
    print(f"  {label}")
    for i, d in enumerate(top, 1):
        print(f"    {i}. {d['meeting']} {d['genre']:<9} {d['bpc']:.4f}")
compare("E2", "Spearman ρ(char transformer, 5-gram) on F")
for label, means in ((CONF_LABEL, CONF["mean_bpc"]),
                     (RUN_LABEL, MEAN_BPC)):
    print(f"  {label:<34} mean BPC on F: char {means['char']['F']:.4f}, "
          f"5-gram {means['ngram']['F']:.4f}")
assert all(d["genre"] == "minutes" for d in CR["E1"])
assert CR["E2"]["point"] < CR["H3"]["point"]
assert CONF["mean_bpc"]["char"]["F"] < CONF["mean_bpc"]["ngram"]["F"]

**Interpretation (exploratory).** E1: all five most surprising F documents are minutes (H2 from the top of the ranking); no causal or market claim. E2: the transformer has lower mean BPC than the 5-gram, yet their rankings are strongly associated, less so than char and BPE (H3): much of the ranking is visible from local character statistics.

## 7. Post-hoc exploration (not pre-registered)

Chosen after seeing the confirmatory results, to locate the H1 contradiction; no tests, intervals or labels. Both runs: (i) mean char BPC by split × genre; (ii) by meeting year × genre; (iii) six highest-BPC statements.

In [ ]:
def mean_table(scores, key):
    """Mean char BPC and document count per (key(score), genre)."""
    groups = {}
    for s in scores:
        groups.setdefault((key(s), s.genre), []).append(s.bpc)
    return {k: (float(np.mean(v)), len(v)) for k, v in sorted(groups.items())}


def print_table(title, table):
    print(title)
    print(f"  {'':>6} {'statement':>16} {'minutes':>16}")
    for row in sorted({r for r, _ in table}):
        cells = (f"{table[row, g][0]:.4f} (n={table[row, g][1]:>2})"
                 if (row, g) in table else "-"
                 for g in ("statement", "minutes"))
        print(f"  {row:>6} " + " ".join(f"{c:>16}" for c in cells))


POSTHOC = {}
for label, scores in ((RUN_LABEL, SCORES["char"]),
                      (CONF_LABEL, CONF_SCORES["char"])):
    by_split = mean_table(scores, lambda s: s.split)
    by_year = mean_table(scores, lambda s: s.meeting[:4])
    top = sorted((s for s in scores if s.genre == "statement"),
                 key=lambda s: -s.bpc)[:6]
    POSTHOC[label] = {"by_split": by_split, "by_year": by_year, "top": top}
    print(f"===== {label}")
    print_table("(i) mean char BPC by split x genre", by_split)
    print_table("(ii) mean char BPC by meeting year x genre", by_year)
    print("(iii) six highest-BPC statements")
    for i, s in enumerate(top, 1):
        print(f"  {i}. {s.meeting} ({s.split}) {s.bpc:.4f}")
    print()
if FAKE_SPLIT:
    print(f"This run is {MODE.upper()} on a fake split: NOT A RESULT.")

c = POSTHOC[CONF_LABEL]
assert max(c["by_year"], key=lambda k: c["by_year"][k][0]) == (
    "2020", "minutes")
assert c["by_split"]["N", "minutes"][0] > c["by_split"]["F", "minutes"][0]
assert c["by_split"]["N", "statement"][0] < c["by_split"]["F", "statement"][0]
assert sum(s.meeting.startswith("2022") for s in c["top"]) > len(c["top"]) / 2

**Reading (post-hoc, descriptive).** In the confirmatory run, F statements score higher than N statements and F minutes lower than N minutes (i), so the minutes carry the negative Δ₁. The 2020 minutes are the most surprising year × genre group (ii). Most of the six most surprising statements are from 2022 (iii). This describes scores, not causes or markets.

## 8. Data source

Board of Governors of the Federal Reserve System, FOMC statements and minutes, https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm. Collected by `build_fomc_corpus.py`, `build_fomc_minutes_corpus.py` and `merge_fomc_corpus.py`; frozen by SHA-256 (section 2).

Limitations, references and the AI-use statement are in README.md.

## 9. Runtime

Writes `runs/<MODE>/results.json` with every number above, then adds the total Run all time.

In [ ]:
RESULTS = {
    "design_version": DESIGN_VERSION, "mode": MODE, "environment": ENV,
    "settings": SETTINGS, "checks": CHECKS, "counts": COUNTS,
    "feasibility": None if FAKE_SPLIT else FEASIBILITY,
    "nf_boundary": str(NF_BOUNDARY), "tokenizers": TOKEN_STATS,
    "training": TRAINING,
    "skipped_updates": {k: v["skipped_updates"] for k, v in TRAINING.items()},
    "samples": {"prompt": PROMPT, **SAMPLES}, "mean_bpc": MEAN_BPC,
    "scores": {k: [dataclasses.asdict(s) for s in v]
               for k, v in SCORES.items()},
    "results": {"H1": H1.as_dict(), "H2": H2.as_dict(), "H3": H3.as_dict(),
                "E1": E1, "E2": E2.as_dict(),
                "bootstrap": {"resamples": analysis.RESAMPLES,
                              "seed": analysis.SEED,
                              "ci_level": analysis.CI_LEVEL}},
    "scoring_projection": SCORING_PROJECTION,
    "posthoc": {
        k: [dataclasses.asdict(s) for s in v] if k == "top" else
        {"|".join(g): {"mean_bpc": m, "n": n} for g, (m, n) in v.items()}
        for k, v in POSTHOC[RUN_LABEL].items()},
    "figures": FIGURES, "times_s": TIMES,
}
RESULTS_PATH = RUN_DIR / "results.json"
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=1) + "\n")
TOTAL_S = time.perf_counter() - T_START  # after the full write
RESULTS["total_runtime_s"] = TOTAL_S
if SCORING_PROJECTION is not None:
    # Measured total with fake-split scoring swapped for the projection.
    # Fits on the smaller fake T (BPE, n-gram, tokenizing) are not rescaled.
    RESULTS["projected_real_total_s"] = TOTAL_S + sum(
        v["projected_s"] - v["measured_s"]
        for v in SCORING_PROJECTION.values())
RESULTS_PATH.write_text(json.dumps(RESULTS, indent=1) + "\n")
for k, v in TIMES.items():
    print(f"{k:>12}: {v:7.1f} s")
print(f"saved {RESULTS_PATH}")
print(f"TOTAL RUNTIME: {TOTAL_S:.1f} s ({TOTAL_S / 60:.2f} min)")
if "projected_real_total_s" in RESULTS:
    print(f"projected real-run total (scoring rescaled to real N/F sizes): "
          f"{RESULTS['projected_real_total_s']:.1f} s "
          f"({RESULTS['projected_real_total_s'] / 60:.2f} min)")
if FAKE_SPLIT:
    print(BANNER)